# 04 — Loading the 20K agent SFT trajectories

The HF dataset ships **20K training trajectories + 20K holdout trajectories**
in ShareGPT format under `agent_sft/`. Each trajectory shows an oracle agent
answering one MC question by calling tools (segment + measure +
lookup_knowledge) and reasoning over the results.

This notebook:
1. Downloads + inspects one trajectory
2. Shows a minimal LoRA fine-tuning loop on the 20K trainset (1 step,
   just to verify the data plugs into HF TRL)


## 1. Download just the agent_sft folder


In [ ]:
from huggingface_hub import snapshot_download

root = snapshot_download(
    "tumor-vqa/DeepTumorVQA_2.0",
    repo_type="dataset",
    allow_patterns=["agent_sft/**"],
)
print(f"agent_sft folder: {root}/agent_sft")


## 2. Inspect one trajectory


In [ ]:
import json
from pathlib import Path

with open(f"{root}/agent_sft/train.jsonl") as f:
    sample = json.loads(f.readline())

print("Keys:", list(sample.keys()))
print(f"\n# turns: {len(sample['conversations'])}")
print(f"# tools available: {len(json.loads(sample['tools']))}")
print()
for turn in sample["conversations"][:8]:
    role = turn["from"]
    text = turn["value"]
    print(f"--- [{role}] ({len(text)} chars) ---")
    print(text[:300] + ("..." if len(text) > 300 else ""))
    print()


## 3. Coverage note

The 20K trajectories cover **283 unique training images** (a subset of the
8,334-image train pool). They span **38 of 42 task subtypes** — the 4
excluded subtypes need spatial information not provided by the available
tools. See `agent_sft/coverage_note.md` for details.

Extending to the full train pool is planned for v2.1.


## 4. Minimal LoRA fine-tuning loop (TRL)

This is a one-step proof of usability — not a real training run.


In [ ]:
# pip install -q transformers peft trl datasets bitsandbytes  # uncomment if needed

from datasets import load_dataset
ds = load_dataset("json", data_files=f"{root}/agent_sft/train.jsonl",
                  split="train")
print(f"Loaded {len(ds)} trajectories")
print(f"First example keys: {list(ds[0].keys())}")


In [ ]:
# Convert ShareGPT -> single chat-template string
# (your real training script would batch-process this)
example = ds[0]
print("Trajectory has", len(example["conversations"]), "turns:")
for turn in example["conversations"]:
    print(f"  [{turn['from']:14s}] {turn['value'][:80]}...")


From here, a real training run would:

1. Apply the model's chat template to the conversations field
2. Mask losses on user / observation / function_call observation turns
   (only train on `gpt` / `function_call` model-generated content)
3. Plug into TRL's `SFTTrainer` with PEFT/LoRA

See the paper's "Meissa SFT (oracle)" row for what this gets you on the
benchmark: **63.8% Overall** (+25.6 pp over zero-shot Meissa-4B).
